# ResNet50 Enhanced MLflow Tracking - v5.0
## Sistema completo de monitoreo y registro

### 🚀 Nuevas características v5.0:
- 📊 **Métricas del sistema**: CPU, memoria, disco, MPS
- 🔬 **MLflow completo**: Experimentos organizados por trials
- 📈 **Registro detallado**: Métricas por época, fase y trial
- 🏷️ **Tags avanzados**: Metadatos completos de cada experimento
- 💾 **Artefactos**: Modelos con signatura, predicciones, matrices confusión

# Ejecutar con kernel NPU4 o superior

# Resnet 
## Version Apple Mx
#### Estos cambios permiten usar la GPU MPS de Apple

Se configura VSCode para que use automaticamente, si es posible, GPUs
Configuracion : @id:editor.experimentalGpuAcceleration @id:terminal.integrated.gpuAcceleration -> on

Se reemplaza esta linea:

-----------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

-----------------------------------------------

por esto:

-----------------------------------------------

### Configuración optimizada para Mac M4
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✅ Usando aceleración MPS (GPU M4)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("✅ Usando CUDA")
else:
    device = torch.device("cpu")
    print("⚠️ Usando CPU solamente")

print(f"Dispositivo seleccionado: {device}")

-----------------------------------------------

Finalmente, se cambia:

-----------------------------------------------

epoch_acc = epoch_phase_running_corrects.double() / len(datasets[phase])

-----------------------------------------------

por esto otro:

-----------------------------------------------

### Conversión compatible con MPS

if device.type == 'mps':
    epoch_acc = epoch_phase_running_corrects.float() / len(datasets[phase])
else:
    epoch_acc = epoch_phase_running_corrects.double() / len(datasets[phase])

-----------------------------------------------

In [ ]:
# ===== CONFIGURACIÓN INICIAL V5.0 =====
import time
import warnings
warnings.filterwarnings("ignore")

# Configuración del experimento
proyecto = 'petfinder'
experimento = 'enhanced_v5'  # Nueva versión con tracking completo
ciclos = 2

print("🚀 RESNET50 ENHANCED TRACKING v5.0")
print(f"   ├── Proyecto: {proyecto}")
print(f"   ├── Experimento: {experimento}")
print(f"   └── Ciclos: {ciclos}")

In [ ]:
# ===== SISTEMA DE MONITOREO DE RECURSOS V5.0 =====
import psutil
import threading
import numpy as np

class SystemMonitor:
    """Monitor de recursos del sistema para MLflow"""
    
    def __init__(self):
        self.monitoring = False
        self.metrics = {
            'cpu_percent': [],
            'memory_percent': [],
            'memory_used_gb': [],
            'disk_usage_percent': [],
            'timestamps': []
        }
        
        # Métricas específicas para MPS/CUDA si disponible
        if torch.backends.mps.is_available():
            self.metrics['device_type'] = 'mps'
        elif torch.cuda.is_available():
            self.metrics['device_type'] = 'cuda'
            self.metrics['gpu_memory_allocated'] = []
            self.metrics['gpu_memory_cached'] = []
        else:
            self.metrics['device_type'] = 'cpu'
    
    def start_monitoring(self):
        """Inicia el monitoreo en background"""
        self.monitoring = True
        self.monitor_thread = threading.Thread(target=self._monitor_loop, daemon=True)
        self.monitor_thread.start()
        print("📊 Monitoreo de recursos iniciado")
    
    def stop_monitoring(self):
        """Detiene el monitoreo"""
        self.monitoring = False
        print("📊 Monitoreo de recursos detenido")
    
    def _monitor_loop(self):
        """Loop principal de monitoreo"""
        while self.monitoring:
            try:
                # Métricas básicas del sistema
                cpu_percent = psutil.cpu_percent(interval=1)
                memory = psutil.virtual_memory()
                disk = psutil.disk_usage('/')
                
                self.metrics['cpu_percent'].append(cpu_percent)
                self.metrics['memory_percent'].append(memory.percent)
                self.metrics['memory_used_gb'].append(memory.used / (1024**3))
                self.metrics['disk_usage_percent'].append(disk.percent)
                self.metrics['timestamps'].append(time.time())
                
                # Métricas GPU si CUDA disponible
                if torch.cuda.is_available():
                    gpu_allocated = torch.cuda.memory_allocated() / (1024**3)  # GB
                    gpu_cached = torch.cuda.memory_reserved() / (1024**3)  # GB
                    self.metrics['gpu_memory_allocated'].append(gpu_allocated)
                    self.metrics['gpu_memory_cached'].append(gpu_cached)
                    
            except Exception as e:
                print(f"⚠️ Error en monitoreo: {e}")
            
            time.sleep(2)  # Monitorear cada 2 segundos
    
    def get_summary_metrics(self):
        """Obtiene métricas resumidas para MLflow"""
        if not self.metrics['cpu_percent']:
            return {}
            
        summary = {
            'system_cpu_avg': np.mean(self.metrics['cpu_percent']),
            'system_cpu_max': np.max(self.metrics['cpu_percent']),
            'system_memory_avg_percent': np.mean(self.metrics['memory_percent']),
            'system_memory_max_percent': np.max(self.metrics['memory_percent']),
            'system_memory_avg_gb': np.mean(self.metrics['memory_used_gb']),
            'system_memory_max_gb': np.max(self.metrics['memory_used_gb']),
            'system_disk_usage_percent': np.mean(self.metrics['disk_usage_percent']),
            'device_type': self.metrics['device_type']
        }
        
        # Métricas GPU si disponible
        if 'gpu_memory_allocated' in self.metrics:
            summary['gpu_memory_avg_gb'] = np.mean(self.metrics['gpu_memory_allocated'])
            summary['gpu_memory_max_gb'] = np.max(self.metrics['gpu_memory_allocated'])
            summary['gpu_cache_avg_gb'] = np.mean(self.metrics['gpu_memory_cached'])
            summary['gpu_cache_max_gb'] = np.max(self.metrics['gpu_memory_cached'])
        
        return summary
    
    def reset_metrics(self):
        """Resetea las métricas acumuladas"""
        for key in self.metrics:
            if isinstance(self.metrics[key], list):
                self.metrics[key].clear()

# Crear monitor global
system_monitor = SystemMonitor()
print("✅ Sistema de monitoreo configurado")

In [1]:
# ===== IMPORTS COMPLETOS V5.0 =====
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, cohen_kappa_score
import os
import shutil
import time
import copy
import datetime
from tqdm import tqdm
import logging
import sys

import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

import torch
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.autograd import Variable
import torch.nn.functional as F

from joblib import load, dump

# MLflow para tracking de experimentos
import mlflow
import mlflow.pytorch

from utils import plot_confusion_matrix

print("✅ Imports completados para v5.0")

✅ Imports completados para v5.0


In [5]:
# Se agrega este codigo para evitar warnings de optuna
import time
import warnings
warnings.filterwarnings("ignore")

# Configuración adicional del experimento
proyecto = 'petfinder' # No cambiar nombre porque se usa para la base de optun
experimento = 'original' # Este nombre se cambia para describir el experimento

# epochs -> ciclos
ciclos = 2


In [ ]:
# ===== CONFIGURACIÓN DE LOGGING PROFESIONAL =====
import logging
import sys
from datetime import datetime
import os

def setup_professional_logging(proyecto, experimento):
    """
    Configura un sistema de logging profesional para el experimento de ML
    
    Args:
        proyecto (str): Nombre del proyecto
        experimento (str): Nombre del experimento específico
    
    Returns:
        logging.Logger: Logger configurado
    """
    
    # Crear directorio de logs si no existe
    log_dir = f"../logs/{proyecto}"
    os.makedirs(log_dir, exist_ok=True)
    
    # Crear nombre único para el archivo de log
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_filename = f"{log_dir}/{experimento}_{timestamp}.log"
    
    # Configurar formato detallado
    detailed_formatter = logging.Formatter(
        '%(asctime)s | %(name)s | %(levelname)8s | %(funcName)s:%(lineno)d | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    # Configurar formato para consola (más compacto)
    console_formatter = logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s',
        datefmt='%H:%M:%S'
    )
    
    # Crear logger principal
    logger_name = f"{proyecto}_{experimento}"
    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.INFO)
    
    # Limpiar handlers previos si existen
    logger.handlers.clear()
    
    # Handler para archivo (nivel DEBUG - todo)
    file_handler = logging.FileHandler(log_filename, mode='w', encoding='utf-8')
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(detailed_formatter)
    logger.addHandler(file_handler)
    
    # Handler para consola (nivel INFO - solo información importante)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(console_formatter)
    logger.addHandler(console_handler)
    
    # Evitar propagación a logger raíz
    logger.propagate = False
    
    # Log inicial del sistema
    logger.info("="*80)
    logger.info(f"🚀 INICIANDO EXPERIMENTO: {proyecto} - {experimento}")
    logger.info(f"📁 Log file: {log_filename}")
    logger.info(f"🕐 Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    logger.info("="*80)
    
    return logger

# Configurar logging para este experimento
logger = setup_professional_logging(proyecto, experimento)

# Log de configuración inicial
logger.info(f"📊 Configuración del experimento:")
logger.info(f"   ├── Proyecto: {proyecto}")
logger.info(f"   ├── Experimento: {experimento}")
logger.info(f"   └── Ciclos de entrenamiento: {ciclos}")

# Log del entorno de ejecución
logger.debug(f"🐍 Python version: {sys.version}")
logger.debug(f"💻 Working directory: {os.getcwd()}")
logger.debug(f"⚠️  Warnings filtrados: Activo")

### **FUENTES**:

PetFinder Kaggle:

https://www.kaggle.com/competitions/petfinder-adoption-prediction/data

First Tutorial:

https://towardsdatascience.com/how-to-train-an-image-classifier-in-pytorch-and-use-it-to-perform-basic-inference-on-single-images-99465a1e9bf5

---> no esta disponible <---


Second Deep Tutorial:

https://rumn.medium.com/part-1-ultimate-guide-to-fine-tuning-in-pytorch-pre-trained-model-and-its-configuration-8990194b71e

Logo Recognition API:

https://heartbeat.comet.ml/logo-recognition-ios-application-using-machine-learning-and-flask-api-aec4eff3be11

Hybrid (multimodal) neural network architecture : Combination of tabular, textual and image inputs:

https://medium.com/@dave.cote.msc/hybrid-multimodal-neural-network-architecture-combination-of-tabular-textual-and-image-inputs-7460a4f82a2e



### **INDICACIONES PREVIAS**:

+ **Git**:
    + Clonamos el repo: root de todos los repos y ponemos git clone "url_repo"
    + Hacemos el checkout de la rama main: git checkout -b new-branch

+ **Poetry**:
    + Instalamos poetry: https://python-poetry.org/docs/
    + Realizamos un Update del pyproject: poetry update
    + Activamos el entorno que creo poetry: poetry shell --> no soportado
    + Intentamos correr una celda, si nos pide seleccionar el environment y no lo vemos en la lista, cerrar y volver abrir VSC

+ **Torch y CUDA**:
    + Verificar que versión pide torch:
        + Versión de torch instalada: poetry show (en mi caso la 1.13.1)
        + Buscar la versión correspondiente en la documentación: https://pytorch.org/get-started/previous-versions/  (en mi caso el 11.7)
    + Instalar CUDA para Torch (buscar la versión correspondiente de CUDA): https://developer.nvidia.com/cuda-11-7-0-download-archive
    + Verificar que CUDA esté funcional: correr en una celda torch.cuda.is_available()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, cohen_kappa_score
import os
import shutil
import time
import copy
import datetime
from tqdm import tqdm

import cv2
import matplotlib.pyplot as plt

import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

import torch
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.autograd import Variable
import torch.nn.functional as F

from joblib import load, dump

# MLflow para tracking de experimentos
import mlflow
import mlflow.pytorch
import mlflow.optuna

from utils import plot_confusion_matrix

# ===== CONFIGURACIÓN DE MLFLOW =====
# Configurar MLflow tracking
mlflow_tracking_uri = f"file://{os.path.abspath('../work')}/mlruns"
mlflow.set_tracking_uri(mlflow_tracking_uri)

# Configurar experimento MLflow
experiment_name = f"{proyecto}_{experimento}"
try:
    experiment = mlflow.create_experiment(experiment_name)
    logger.info(f"🆕 Experimento MLflow creado: {experiment_name}")
except mlflow.exceptions.MlflowException:
    experiment = mlflow.get_experiment_by_name(experiment_name)
    logger.info(f"📂 Experimento MLflow existente: {experiment_name}")

mlflow.set_experiment(experiment_name)
logger.info(f"🔬 MLFLOW CONFIGURADO")
logger.info(f"   ├── Tracking URI: {mlflow_tracking_uri}")
logger.info(f"   ├── Experimento: {experiment_name}")
logger.info(f"   └── Experiment ID: {experiment.experiment_id if hasattr(experiment, 'experiment_id') else 'N/A'}")

# Verificamos que CUDA está funcional
print(f'Disponibilidad CUDA: {torch.cuda.is_available()}')
print(f'Disponibilidad MPS: {torch.backends.mps.is_available()}')

# Logging detallado de las capacidades del sistema
logger.info("🔍 ANÁLISIS DEL SISTEMA DE CÓMPUTO")
logger.info(f"   ├── PyTorch version: {torch.__version__}")
logger.info(f"   ├── CUDA disponible: {torch.cuda.is_available()}")
logger.info(f"   ├── MPS disponible: {torch.backends.mps.is_available()}")
logger.info(f"   └── Núcleos CPU: {os.cpu_count()}")

if torch.cuda.is_available():
    logger.debug(f"   ├── CUDA version: {torch.version.cuda}")
    logger.debug(f"   ├── GPUs disponibles: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        logger.debug(f"   ├── GPU {i}: {torch.cuda.get_device_name(i)}")
        logger.debug(f"   └── Memoria GPU {i}: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f}GB")

**Seteo el Modelo**

Teoría de Resnet: https://towardsdatascience.com/introduction-to-resnets-c0a830a288a4

In [ ]:
# Importo modelo ResNet entrenado en Imagenet
logger.info("🧠 CONFIGURACIÓN DEL MODELO")
logger.info("   ├── Cargando ResNet50 preentrenado...")

resnet50 = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
logger.info("   ├── ✅ ResNet50 cargado exitosamente")

# Modificar la última capa para adaptarse a tu problema específico
num_ftrs = resnet50.fc.in_features
resnet50.fc = torch.nn.Linear(num_ftrs, 5) # Clasificación 5 clases
logger.info(f"   ├── Capa final modificada: {num_ftrs} → 5 clases")

# Configuro para usar cuda si está disponible

# Configuración optimizada para Mac Mx
logger.info("   ├── Detectando dispositivo óptimo...")

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✅ Usando aceleración MPS (GPU Mx Apple)")
    logger.info("   ├── 🚀 Dispositivo seleccionado: MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("✅ Usando CUDA")
    logger.info(f"   ├── 🚀 Dispositivo seleccionado: CUDA (GPU {torch.cuda.get_device_name()})")
else:
    device = torch.device("cpu")
    print("⚠️ Usando CPU solamente")
    logger.warning("   ├── ⚠️  Dispositivo seleccionado: CPU (sin aceleración)")

print(f"Dispositivo seleccionado: {device}")
logger.info(f"   └── Dispositivo final: {device}")

logger.info("   ├── Transfiriendo modelo al dispositivo...")
resnet50 = resnet50.to(device)
logger.info("   ├── ✅ Modelo transferido exitosamente")

# Instancio del criterio de pérdida CrossEntropyLoss
criterion = nn.CrossEntropyLoss()
logger.info("   └── ✅ Criterio de pérdida configurado: CrossEntropyLoss")



**Seteo parámetros, directorios y funciones**

In [ ]:
# Paths
BASE_DIR = '../'
PATH_TO_TRAIN = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train/train.csv")
PATH_TO_IMAGES_DIR = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train_images")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, f"work/{proyecto}/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, f"work/{proyecto}/optuna_artifacts")

MODEL_NAME = '05 ResNet'
MODEL_VERSION = '1.0.0'

# Parametros y variables
CREATE_PYTORCH_DIRECTORIES = 1
SEED = 42
BATCH_SIZE = 50
TEST_SIZE = 0.2
IMAGE_SIZE = 299
CPU_CORES = os.cpu_count()

# Logging de configuración de parámetros
logger.info("⚙️  CONFIGURACIÓN DE PARÁMETROS")
logger.info(f"   ├── Modelo: {MODEL_NAME} v{MODEL_VERSION}")
logger.info(f"   ├── Batch size: {BATCH_SIZE}")
logger.info(f"   ├── Test size: {TEST_SIZE} ({TEST_SIZE*100}%)")
logger.info(f"   ├── Tamaño de imagen: {IMAGE_SIZE}x{IMAGE_SIZE}")
logger.info(f"   ├── Semilla aleatoria: {SEED}")
logger.info(f"   ├── CPU cores disponibles: {CPU_CORES}")
logger.info(f"   └── Crear directorios PyTorch: {'Sí' if CREATE_PYTORCH_DIRECTORIES else 'No'}")

logger.info("📁 CONFIGURACIÓN DE RUTAS")
logger.info(f"   ├── BASE_DIR: {BASE_DIR}")
logger.info(f"   ├── CSV de entrenamiento: {PATH_TO_TRAIN}")
logger.info(f"   ├── Directorio de imágenes: {PATH_TO_IMAGES_DIR}")
logger.info(f"   ├── Archivos temporales: {PATH_TO_TEMP_FILES}")
logger.info(f"   └── Artefactos Optuna: {PATH_TO_OPTUNA_ARTIFACTS}")

# Verificar que archivos/directorios existen
logger.debug("🔍 Verificando existencia de archivos críticos:")
if os.path.exists(PATH_TO_TRAIN):
    logger.debug(f"   ├── ✅ CSV encontrado: {PATH_TO_TRAIN}")
else:
    logger.error(f"   ├── ❌ CSV NO encontrado: {PATH_TO_TRAIN}")
    
if os.path.exists(PATH_TO_IMAGES_DIR):
    logger.debug(f"   └── ✅ Directorio de imágenes encontrado: {PATH_TO_IMAGES_DIR}")
else:
    logger.error(f"   └── ❌ Directorio de imágenes NO encontrado: {PATH_TO_IMAGES_DIR}")

# Armo el nuevo directorio de train
new_train_directory = os.path.join(BASE_DIR, 'work/train_images_classes')
os.makedirs(new_train_directory, exist_ok=True) # si ya existe el nombre, lo deja como está

# Armo el nuevo directorio de validación
new_val_directory = os.path.join(BASE_DIR, 'work/val_images_classes')
os.makedirs(new_val_directory, exist_ok=True)

# Crear directorios necesarios para artefactos
os.makedirs(PATH_TO_TEMP_FILES, exist_ok=True)
os.makedirs(PATH_TO_OPTUNA_ARTIFACTS, exist_ok=True)

logger.info("📂 DIRECTORIOS DE TRABAJO CREADOS")
logger.info(f"   ├── Entrenamiento: {new_train_directory}")
logger.info(f"   ├── Validación: {new_val_directory}")
logger.info(f"   ├── Archivos temporales: {PATH_TO_TEMP_FILES}")
logger.info(f"   └── Artefactos Optuna: {PATH_TO_OPTUNA_ARTIFACTS}")

# Definir las clases ordenadas
class_names = ['0', '1', '2', '3', '4']
logger.info(f"🏷️  CLASES DEFINIDAS: {class_names}")

# Mapear las etiquetas de las clases a números enteros consecutivos
class_to_idx = {class_name: i for i, class_name in enumerate(class_names)}

# Creo las carpetas de clases dentro de los directorios
for clase in class_names: # Una para cada clase
   os.makedirs(os.path.join(new_train_directory, str(clase)), exist_ok=True)
   os.makedirs(os.path.join(new_val_directory, str(clase)), exist_ok=True)




# Funciones para la carga y el preproceso
def resize_to_square(im):
    old_size = im.shape[:2] # old_size is in (height, width) format
    # Calcula el factor de escala necesario para redimensionar la imagen de manera que el lado más largo tenga el tamaño deseado 
    ratio = float(IMAGE_SIZE)/max(old_size)
    # Calcula las nuevas dimensiones de la imagen 
    new_size = tuple([int(x*ratio) for x in old_size])
    # Redimensiona la imagen con el nuevo tamaño
    im = cv2.resize(im, (new_size[1], new_size[0]))
    # Calcula las diferencias de tamaño y agrega pixeles (color negro) en los extremos para que quede centrada y cuadrada 
    delta_w = IMAGE_SIZE - new_size[1]
    delta_h = IMAGE_SIZE - new_size[0]
    top, bottom = delta_h//2, delta_h-(delta_h//2)
    left, right = delta_w//2, delta_w-(delta_w//2)
    color = [0, 0, 0]
    new_image = cv2.copyMakeBorder(im, top, bottom, left, right, cv2.BORDER_CONSTANT,value=color)
    return new_image


def load_image(pet_id):
    path_to_image = os.path.join(PATH_TO_IMAGES_DIR, f'{pet_id}-1.jpg') # Irá a la primera imagen de la mascota
    image = cv2.imread(path_to_image)
    # Convierte la imagen de BGR a RGB porque estos modelos esperan ese orden de canales
    image = cv2.convertScaleAbs(image)
    image= cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    new_image = resize_to_square(image)
    return new_image


In [ ]:

def visualize_pet(pet_id):
    path_to_image = os.path.join(PATH_TO_IMAGES_DIR, f'{pet_id}-1.jpg') # Irá a la primera imagen de la mascota
    # Cargar la imagen
    image_to_show = cv2.imread(path_to_image)
    # Convertir a formato RGB
    image_to_show = cv2.cvtColor(image_to_show, cv2.COLOR_BGR2RGB)
    # Visualizar la imagen
    plt.imshow(image_to_show)
    plt.axis('off')  # No mostrar los ejes
    plt.show()

def visualize_image(image):
    # Convierte la imagen a un formato de enteros (CV_8U)
    image = cv2.convertScaleAbs(image)
    image= cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    # Visualizar la imagen
    plt.imshow(image.astype(np.uint8))
    plt.axis('off')  # No mostrar los ejes
    plt.show()


**Cargo y Proceso Data**

Nota: Pytorch necesita que estén las imágenes en los distintos directorios según su clase y su participación en el training

In [ ]:
# Cargo
train_df = pd.read_csv(PATH_TO_TRAIN)

# Split para validación
train_data, val_data = train_test_split(train_df,
                               test_size = TEST_SIZE,
                               random_state = SEED,
                               stratify = train_df.AdoptionSpeed)

# if CREATE_PYTORCH_DIRECTORIES == 1:  # Ya fue ejecutado al menos una vez, por lo que las carpetas existen
if CREATE_PYTORCH_DIRECTORIES == 0: # Poner en 0 si ya tengo las carpetas train_images_classes y val_images_classes con las imágenes copiadas
    # Función para copiar las imágenes a los directorios correspondientes
    def copy_imag(data, directorio_destino):
        for index, row in data.iterrows():
            petID = row['PetID']
            adoption_speed = row['AdoptionSpeed']
            
            # Nombre del archivo de imagen
            nombre_archivo = f"{petID}-1.jpg"
            
            # Ruta completa de la imagen de origen
            ruta_origen = os.path.join(PATH_TO_IMAGES_DIR, nombre_archivo)
            
            # Ruta completa del directorio de destino
            ruta_destino = os.path.join(directorio_destino, str(adoption_speed), nombre_archivo)
            
            # Verificar si el archivo de origen existe
            if os.path.exists(ruta_origen):
                # Copiar el archivo de origen al directorio de destino
                shutil.copy2(ruta_origen, ruta_destino)
        print("Completada la copia a: ",str(directorio_destino))

    # Copiar las imágenes al directorio de train
    copy_imag(train_data, new_train_directory)

    # Copiar las imágenes al directorio de val
    copy_imag(val_data, new_val_directory)

    print("Proceso completado.")

In [ ]:
# Genero los DataLoaders
def create_dataloaders(train_directory, val_directory, batch_size, num_workers):
    # Transformaciones de imagen para el conjunto de entrenamiento
    train_transforms = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    # Transformaciones de imagen para el conjunto de validación (sin data augment)
    val_transforms = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    # Crear conjuntos de datos para el conjunto de entrenamiento y validación
    conjunto_entrenamiento = datasets.ImageFolder(train_directory, transform=train_transforms)
    conjunto_validacion = datasets.ImageFolder(val_directory, transform=val_transforms)

    # Asignar las clases ordenadas al conjunto de datos
    conjunto_entrenamiento.class_to_idx = {class_name: i for i, class_name in enumerate(class_names)}
    conjunto_validacion.class_to_idx = {class_name: i for i, class_name in enumerate(class_names)}

    # Crear dataloaders para el conjunto de entrenamiento y validación
    train_dataloader = torch.utils.data.DataLoader(conjunto_entrenamiento, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_dataloader = torch.utils.data.DataLoader(conjunto_validacion, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    return train_dataloader, val_dataloader

# Aplico las funcion de los DataLoaders
logger.info("🔄 CREANDO DATALOADERS")
logger.info(f"   ├── Directorio entrenamiento: {new_train_directory}")
logger.info(f"   ├── Directorio validación: {new_val_directory}")
logger.info(f"   ├── Batch size: {BATCH_SIZE}")
logger.info(f"   └── Workers: {CPU_CORES}")

train_dataloader, val_dataloader = create_dataloaders(new_train_directory , new_val_directory , BATCH_SIZE, CPU_CORES)

logger.info("   ├── ✅ DataLoaders creados exitosamente")
logger.info(f"   ├── Batches de entrenamiento: {len(train_dataloader)}")
logger.info(f"   ├── Batches de validación: {len(val_dataloader)}")
logger.info(f"   ├── Muestras de entrenamiento: {len(train_dataloader.dataset)}")
logger.info(f"   └── Muestras de validación: {len(val_dataloader.dataset)}")

In [ ]:
#Genero una lista de PetIDs con imagen en el orden en que aparecen en el data loader
test_sample_ids = [i[0].split('/')[-1].split('-')[0] for i in val_dataloader.dataset.samples]

logger.info("🆔 IDs DE MUESTRAS EXTRAÍDOS")
logger.info(f"   ├── Total IDs extraídos: {len(test_sample_ids)}")
logger.info(f"   ├── Primeros 5 IDs: {test_sample_ids[:5]}")
logger.info(f"   └── Últimos 5 IDs: {test_sample_ids[-5:]}")

**Entreno**

In [ ]:
def train_val(model, criterion, dataloaders, datasets, device, num_epochs=20, lr=0.001, momentum = 0.9 ,trial=None):
    
    # Log del inicio del entrenamiento
    trial_info = f"Trial {trial.number}" if trial else "Entrenamiento único"
    logger.info("="*80)
    logger.info(f"🚀 INICIANDO ENTRENAMIENTO - {trial_info}")
    logger.info(f"   ├── Learning rate: {lr}")
    logger.info(f"   ├── Momentum: {momentum}")
    logger.info(f"   ├── Épocas: {num_epochs}")
    logger.info(f"   ├── Dispositivo: {device}")
    logger.info(f"   ├── Tamaño entrenamiento: {len(datasets['train'])} muestras")
    logger.info(f"   └── Tamaño validación: {len(datasets['val'])} muestras")
    
    # Instancio Stochastic Gradient Descent (SGD): Defino el parámetro del Learning Rate (define "el paso" en que avanzan los pesos en cada iteración) y el Momentum (pone innercia a la dirección del gradiente descendiente para que no cambie de dirección en minimos locales)
    optimizer = optim.SGD(resnet50.parameters(), lr=lr, momentum=momentum) # Parámetros default del SGD
    logger.debug(f"   ├── Optimizador configurado: SGD(lr={lr}, momentum={momentum})")
    
    #Inicializo variables
    since = time.time()
    logger.debug(f"   └── Timer iniciado: {datetime.datetime.fromtimestamp(since).strftime('%H:%M:%S')}")

    #Inicializo variable de mejor kappa entre trials
    try:
        #Intento obtener el mejor kappa de optuna
        previous_best = study.best_value
        logger.debug(f"   ├── Mejor kappa previo: {previous_best:.4f}")
    except:
        #Si no hay, seteo -999
        previous_best = -999
        logger.debug(f"   ├── Sin kappa previo, usando: {previous_best}")

    #Inicializo variables de mejor modelo y mejor accuracy y mejor kappa de este trial
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    best_kappa =  -999
    logger.debug(f"   └── Variables de mejor modelo inicializadas")

    # ===== MLFLOW: Log parámetros del entrenamiento =====
    if trial is not None:
        # Si es trial de Optuna, crear run child
        run_name = f"trial_{trial.number}"
        tags = {"trial_number": trial.number, "optimizer": "SGD", "model": "ResNet50"}
        use_nested = True
    else:
        # Si es entrenamiento único, usar run existente o crear nested
        run_name = f"single_run_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
        tags = {"optimizer": "SGD", "model": "ResNet50", "type": "single_run"}
        use_nested = True  # Siempre usar nested para evitar conflictos
    
    with mlflow.start_run(run_name=run_name, tags=tags, nested=use_nested):
        # Log parámetros
        mlflow.log_param("learning_rate", lr)
        mlflow.log_param("momentum", momentum)
        mlflow.log_param("num_epochs", num_epochs)
        mlflow.log_param("batch_size", BATCH_SIZE)
        mlflow.log_param("device", str(device))
        mlflow.log_param("train_samples", len(datasets['train']))
        mlflow.log_param("val_samples", len(datasets['val']))
        mlflow.log_param("image_size", IMAGE_SIZE)
        
        if trial:
            mlflow.log_param("trial_number", trial.number)
            mlflow.log_param("optuna_study", trial.study.study_name)


        for epoch in range(num_epochs):
            epoch_start_time = time.time()
            
            logger.info(f"")
            logger.info(f"📊 ÉPOCA {epoch+1}/{num_epochs}")
            logger.info(f"   └── Tiempo actual: {datetime.datetime.now().strftime('%H:%M:%S')}")
            
            print('Epoch {}/{}'.format(epoch, num_epochs - 1))
            print('-' * 10)
            
            #Inicializo listas de kappa true y predicted y scores para esta epoch
            epoch_kappa_labels_true = []
            epoch_kappa_labels_predicted = []
            epoch_output_scores = []

            #Cada epoch tiene una fase de entrenamiento y validación
            for phase in ['train', 'val']:
                phase_start_time = time.time()
                
                if phase == 'train':
                    model.train()  # Set model to training mode
                    logger.debug(f"      ├── 🏋️  Fase ENTRENAMIENTO iniciada")
                else:
                    model.eval()   # Set model to evaluate mode
                    logger.debug(f"      ├── 🧪 Fase VALIDACIÓN iniciada")

                #Inicializo variables de loss y accuracy para esta fase de epoch
                epoch_phase_running_loss = 0.0
                epoch_phase_running_corrects = 0
                
                # Variables para tracking detallado
                batch_count = 0
                total_batches = len(dataloaders[phase])
                log_interval = max(1, total_batches // 10)  # Log cada 10% del progreso

                # Itero sobre los datos.
                for inputs, labels in tqdm(dataloaders[phase]):
                    batch_count += 1
                    
                    inputs = inputs.to(device)
                    labels = labels.to(device)

                    # Zero the parameter gradients
                    optimizer.zero_grad()

                    # Forward
                    # Track history if only in train
                    with torch.set_grad_enabled(phase == 'train'):
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = criterion(outputs, labels)

                        # Backward + optimize only if in training phase
                        if phase == 'train':
                            loss.backward()
                            optimizer.step()
                        elif phase == 'val':
                            #Agrego los valores de kappa true y predicted para cada batch en validación
                            epoch_kappa_labels_true.extend(labels.cpu().numpy().tolist())
                            epoch_kappa_labels_predicted.extend(preds.cpu().numpy().tolist())
                            outputs_np = outputs.cpu().numpy()
                            epoch_output_scores.extend([outputs_np[i,:] for i in range(outputs_np.shape[0])])

                    # Statistics for each phase
                    epoch_phase_running_loss += loss.item() * inputs.size(0)
                    epoch_phase_running_corrects += torch.sum(preds == labels.data)
                    
                    # Log progreso periódico por batch
                    if batch_count % log_interval == 0 or batch_count == total_batches:
                        batch_acc = torch.sum(preds == labels.data).float() / inputs.size(0)
                        progress_pct = (batch_count / total_batches) * 100
                        logger.debug(f"         ├── Batch {batch_count}/{total_batches} ({progress_pct:.1f}%) - Loss: {loss.item():.4f}, Acc: {batch_acc:.4f}")
                    
                    #END OF BATCH
                
                epoch_loss = epoch_phase_running_loss / len(datasets[phase])
                # Conversión compatible con MPS
                if device.type == 'mps':
                    epoch_acc = epoch_phase_running_corrects.float() / len(datasets[phase])
                else:
                    epoch_acc = epoch_phase_running_corrects.double() / len(datasets[phase])

                #Calculo el kappa para cada epoch
                if phase == 'train':
                    #overall_train_losses.append(epoch_loss)
                    current_kappa_score = np.nan
                else:
                    #overall_val_losses.append(epoch_loss)
                    current_kappa_score = cohen_kappa_score(epoch_kappa_labels_true,
                                      epoch_kappa_labels_predicted,
                                      weights = 'quadratic')
                
                # Log detallado de métricas por fase
                phase_duration = time.time() - phase_start_time
                logger.info(f"      ├── 📈 {phase.upper()} completado en {phase_duration:.1f}s")
                logger.info(f"      ├── Loss: {epoch_loss:.6f}")
                logger.info(f"      ├── Accuracy: {epoch_acc*100:.2f}%")
                if not np.isnan(current_kappa_score):
                    logger.info(f"      └── Kappa: {current_kappa_score:.6f}")
                else:
                    logger.info(f"      └── Kappa: N/A (entrenamiento)")
                
                # ===== MLFLOW: Log métricas por época y fase =====
                mlflow.log_metric(f"{phase}_loss", epoch_loss, step=epoch)
                mlflow.log_metric(f"{phase}_accuracy", float(epoch_acc), step=epoch)
                mlflow.log_metric(f"{phase}_duration", phase_duration, step=epoch)
                
                if not np.isnan(current_kappa_score):
                    mlflow.log_metric(f"{phase}_kappa", current_kappa_score, step=epoch)
                        
                print(f'{phase.title()} Loss: {epoch_loss:.4f} Acc: {epoch_acc*100:.2f}% Kappa: {current_kappa_score:.3f}')

                # If this is the best Epoch so far -> Deep copy the model
                if phase == 'val' and current_kappa_score > best_kappa:
                    best_acc = epoch_acc
                    best_kappa = current_kappa_score
                    best_model_wts = copy.deepcopy(model.state_dict())
                    
                    logger.info(f"      🏆 ¡NUEVO MEJOR MODELO!")
                    logger.info(f"         ├── Mejor Accuracy: {best_acc*100:.2f}%")
                    logger.info(f"         └── Mejor Kappa: {best_kappa:.6f}")
                    
                    # ===== MLFLOW: Log mejor modelo =====
                    mlflow.log_metric("best_accuracy", float(best_acc))
                    mlflow.log_metric("best_kappa", best_kappa)

                    #Best Epoch within a trial and better than previous trials
                    if trial is not None and best_kappa > previous_best:
                        logger.info(f"         🎯 Mejor que trials anteriores! (prev: {previous_best:.6f})")

                        #Save test dataset with predictions
                        predicted_filename = os.path.join(PATH_TO_TEMP_FILES,f'test_{trial.study.study_name}_{trial.number}.joblib')
                        predicted_df = pd.DataFrame({'PetID':test_sample_ids,
                                    'pred':epoch_output_scores}).merge(val_data, on='PetID')
                        dump(predicted_df, predicted_filename)
                        logger.debug(f"         ├── Predicciones guardadas: {predicted_filename}")

                        #Generate and save CM 
                        cm_filename = os.path.join(PATH_TO_TEMP_FILES,f'cm_{trial.study.study_name}_{trial.number}.jpg')
                        plot_confusion_matrix(epoch_kappa_labels_true,epoch_kappa_labels_predicted).write_image(cm_filename)
                        logger.debug(f"         └── Matriz confusión guardada: {cm_filename}")
                        
                        # ===== MLFLOW: Log artefactos del mejor modelo =====
                        try:
                            mlflow.log_artifact(predicted_filename, "predictions")
                            mlflow.log_artifact(cm_filename, "confusion_matrices")
                            logger.debug(f"         ├── MLflow: Artefactos registrados")
                        except Exception as e:
                            logger.warning(f"         ├── MLflow warning: {str(e)}")

                #END OF PHASE

            # Log resumen de la época
            epoch_duration = time.time() - epoch_start_time
            logger.info(f"   ⏱️  Época {epoch+1} completada en {epoch_duration:.1f}s")
            
            #END OF EPOCH

        time_elapsed = time.time() - since
        
        # Log resumen final del entrenamiento
        logger.info("")
        logger.info("🏁 ENTRENAMIENTO COMPLETADO")
        logger.info(f"   ├── ⏱️  Tiempo total: {time_elapsed//60:.0f}m {time_elapsed%60:.0f}s")
        logger.info(f"   ├── 🎯 Mejor Accuracy: {best_acc*100:.2f}%")
        logger.info(f"   ├── 🏆 Mejor Kappa: {best_kappa:.6f}")
        
        if trial:
            logger.info(f"   └── Trial {trial.number} finalizado")
        else:
            logger.info(f"   └── Entrenamiento único finalizado")
        
        print('Training complete in {:.0f}m {:.0f}s'.format(
            time_elapsed // 60, time_elapsed % 60))
        print('Best val Acc: {:.2f}%'.format(best_acc * 100))

        # Load best model weights
        model.load_state_dict(best_model_wts)
        logger.debug("   └── Mejores pesos del modelo cargados")

        # Save in optuna trial the best test dataset, cm and model weights
        if trial is not None and best_kappa > previous_best:
            logger.info("💾 GUARDANDO ARTEFACTOS DE MEJOR TRIAL")
            
            upload_artifact(trial, predicted_filename, artifact_store)   
            logger.debug(f"   ├── Predicciones subidas: {predicted_filename}")

            upload_artifact(trial, cm_filename, artifact_store)
            logger.debug(f"   ├── Matriz confusión subida: {cm_filename}")

            file_name = f'{MODEL_NAME}_{MODEL_VERSION}_{trial.number}.pth'
            model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
            torch.save(model, model_path) # Podemos guardar solo los pesos si queremos: best_model.state_dict()
            upload_artifact(trial, model_path, artifact_store)
            logger.info(f"   └── Modelo guardado y subido: {model_path}")
            
            # ===== MLFLOW: Log modelo PyTorch =====
            try:
                # Crear ejemplo de entrada para la signatura del modelo
                dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
                
                mlflow.pytorch.log_model(
                    model, 
                    "model",
                    input_example=dummy_input.cpu().numpy()
                )
                mlflow.log_artifact(model_path, "saved_models")
                logger.debug(f"   └── MLflow: Modelo PyTorch registrado con signatura")
            except Exception as e:
                logger.warning(f"   └── MLflow warning: {str(e)}")

            # ===== MLFLOW: Log métricas finales y metadatos =====
            mlflow.log_metric("total_training_time", time_elapsed)
            mlflow.log_metric("final_best_accuracy", float(best_acc))
            mlflow.log_metric("final_best_kappa", best_kappa)
            
            # Log tags adicionales
            mlflow.set_tag("training_completed", "true")
            mlflow.set_tag("device_used", str(device))
            
            if trial:
                mlflow.set_tag("is_optuna_trial", "true")
                mlflow.set_tag("trial_number", trial.number)
            else:
                mlflow.set_tag("is_optuna_trial", "false")

    return model,best_kappa

# ===== ENTRENAMIENTO ÚNICO (SIN OPTUNA) =====
with mlflow.start_run(run_name="single_training_run") as single_run:
    # Log configuración del entrenamiento único
    mlflow.log_param("training_type", "single_run")
    mlflow.log_param("num_epochs", ciclos)
    mlflow.set_tag("is_single_training", "true")
    
    best_model,_ = train_val(resnet50, criterion, 
                           dataloaders={'train': train_dataloader, 
                                        'val': val_dataloader}, 
                           datasets={'train': train_data, 'val': val_data}, 
                           device=device, 
                           num_epochs=ciclos)

# Guardo el modelo
logger.info("💾 GUARDANDO MODELO FINAL")

run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
file_name = f'{MODEL_NAME}_{MODEL_VERSION}_{run_id}.pth'
model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
torch.save(best_model, model_path) # Podemos guardar solo los pesos si queremos: best_model.state_dict()

logger.info(f"   └── ✅ Modelo guardado: {model_path}")
print(f'Modelo guardado en {model_path}')

# ===== MLFLOW: Log modelo final del entrenamiento único =====
with mlflow.start_run(run_id=single_run.info.run_id):
    try:
        # Crear ejemplo de entrada para la signatura del modelo
        dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
        
        mlflow.pytorch.log_model(
            best_model, 
            "final_model",
            input_example=dummy_input.cpu().numpy()
        )
        mlflow.log_artifact(model_path, "saved_models")
        mlflow.set_tag("final_model_saved", "true")
        logger.info(f"   └── ✅ Modelo final registrado en MLflow con signatura")
    except Exception as e:
        logger.warning(f"   └── MLflow warning: {str(e)}")

In [ ]:
artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)


def optuna_train(trial):
    
    logger.info("="*60)
    logger.info(f"🔬 INICIANDO OPTUNA TRIAL #{trial.number}")
    
    epochs = trial.suggest_int('epochs', ciclos, ciclos)
    lr = trial.suggest_float('lr', 0.00001, 0.1, log=True)
    momentum = trial.suggest_float('momentum', 0.0, 0.95)
    
    logger.info(f"   ├── Hiperparámetros sugeridos:")
    logger.info(f"   ├──   Epochs: {epochs}")
    logger.info(f"   ├──   Learning Rate: {lr:.6f}")
    logger.info(f"   └──   Momentum: {momentum:.4f}")

    # ===== MLFLOW: Crear run padre para el trial de Optuna =====
    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}", nested=True) as parent_run:
        # Log parámetros del trial a nivel padre
        mlflow.log_param("optuna_trial_number", trial.number)
        mlflow.log_param("suggested_epochs", epochs)
        mlflow.log_param("suggested_lr", lr)
        mlflow.log_param("suggested_momentum", momentum)
        
        # Tags para identificar
        mlflow.set_tag("optuna_trial", "true")
        mlflow.set_tag("trial_number", trial.number)

        _,best_score = train_val(resnet50, criterion,
                           dataloaders={'train': train_dataloader, 
                                        'val': val_dataloader}, 
                           datasets={'train': train_data, 'val': val_data}, 
                           device=device, 
                           num_epochs=epochs,
                           lr=lr,
                           momentum = momentum,
                           trial=trial)

        # Log resultado final del trial
        mlflow.log_metric("trial_final_kappa", best_score)
        mlflow.set_tag("trial_completed", "true")

    logger.info(f"🏁 TRIAL #{trial.number} COMPLETADO - Kappa final: {best_score:.6f}")
    logger.info("="*60)
    
    return(best_score)

In [ ]:
logger.info("🔍 INICIANDO OPTIMIZACIÓN OPTUNA")
logger.info(f"   ├── Proyecto: {proyecto}")
logger.info(f"   ├── Modelo: {MODEL_NAME}_{MODEL_VERSION}")
logger.info(f"   ├── Base de datos: ../work/{proyecto}/db.sqlite3")
logger.info(f"   └── Número de trials: 20")

study = optuna.create_study(direction='maximize',
                            storage=f"sqlite:///../work/{proyecto}/db.sqlite3",  # Specify the storage URL here.
                            study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
                            load_if_exists = True)

# ===== MLFLOW: Log estudio de Optuna =====
with mlflow.start_run(run_name=f"optuna_study_{MODEL_NAME}_{MODEL_VERSION}") as study_run:
    # Log configuración del estudio
    mlflow.log_param("n_trials", 20)
    mlflow.log_param("direction", "maximize")
    mlflow.log_param("study_name", f'{MODEL_NAME}_{MODEL_VERSION}')
    mlflow.log_param("storage", f"sqlite:///../work/{proyecto}/db.sqlite3")
    
    # Tags para identificar el estudio
    mlflow.set_tag("optuna_study", "true")
    mlflow.set_tag("model_name", MODEL_NAME)
    mlflow.set_tag("model_version", MODEL_VERSION)
    
    logger.info("🚀 Comenzando optimización...")
    
    # Optuna optimization sin callback específico - MLflow tracking se maneja dentro de optuna_train
    study.optimize(optuna_train, n_trials=20)

    # Log resultados finales del estudio
    mlflow.log_metric("best_trial_number", study.best_trial.number)
    mlflow.log_metric("best_study_kappa", study.best_value)
    
    # Log mejores parámetros
    for param_name, param_value in study.best_params.items():
        mlflow.log_param(f"best_{param_name}", param_value)
    
    mlflow.set_tag("study_completed", "true")

logger.info("🎯 OPTIMIZACIÓN COMPLETADA")
logger.info(f"   ├── Mejor trial: #{study.best_trial.number}")
logger.info(f"   ├── Mejor kappa: {study.best_value:.6f}")
logger.info(f"   └── Mejores parámetros: {study.best_params}")

In [ ]:
# ===== FUNCIÓN DE ENTRENAMIENTO ENHANCED CON MLFLOW V5.0 =====
def train_val_enhanced(model, criterion, dataloaders, datasets, device, 
                      num_epochs=20, lr=0.001, momentum=0.9, trial=None):
    """
    Función de entrenamiento con registro completo en MLflow v5.0
    - Métricas del sistema en tiempo real
    - Registro detallado por época y fase
    - Artefactos completos (modelos, predicciones, matrices)
    - Tags y metadatos organizados
    """
    
    # Configurar logging
    if trial is not None:
        print(f\"🔬 TRIAL {trial.number} - Enhanced Tracking\")
        run_name = f\"trial_{trial.number}_enhanced\"
        tags = {
            \"trial_number\": trial.number,
            \"optimizer\": \"SGD\",
            \"model\": \"ResNet50\",
            \"version\": \"v5.0\",
            \"enhanced_tracking\": \"true\"
        }
    else:
        print(f\"🚀 ENTRENAMIENTO ÚNICO - Enhanced Tracking\")
        run_name = f\"single_run_enhanced_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}\"\n        tags = {\n            \"optimizer\": \"SGD\",\n            \"model\": \"ResNet50\",\n            \"type\": \"single_run\",\n            \"version\": \"v5.0\",\n            \"enhanced_tracking\": \"true\"\n        }\n    \n    # Iniciar monitoreo de recursos\n    system_monitor.reset_metrics()\n    system_monitor.start_monitoring()\n    \n    # Configurar optimizador\n    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)\n    \n    # Variables de control\n    since = time.time()\n    best_model_wts = copy.deepcopy(model.state_dict())\n    best_acc = 0.0\n    best_kappa = -999\n    \n    # Obtener mejor kappa previo si existe\n    try:\n        previous_best = study.best_value if trial else -999\n    except:\n        previous_best = -999\n    \n    # ===== MLFLOW: Iniciar run con registro completo =====\n    with mlflow.start_run(run_name=run_name, tags=tags, nested=True):\n        \n        # === PARÁMETROS DE CONFIGURACIÓN ===\n        mlflow.log_param(\"learning_rate\", lr)\n        mlflow.log_param(\"momentum\", momentum)\n        mlflow.log_param(\"num_epochs\", num_epochs)\n        mlflow.log_param(\"batch_size\", dataloaders['train'].batch_size)\n        mlflow.log_param(\"device\", str(device))\n        mlflow.log_param(\"train_samples\", len(datasets['train']))\n        mlflow.log_param(\"val_samples\", len(datasets['val']))\n        mlflow.log_param(\"image_size\", 299)  # IMAGE_SIZE\n        mlflow.log_param(\"model_architecture\", \"ResNet50\")\n        mlflow.log_param(\"num_classes\", 5)\n        mlflow.log_param(\"criterion\", \"CrossEntropyLoss\")\n        \n        if trial:\n            mlflow.log_param(\"trial_number\", trial.number)\n            mlflow.log_param(\"optuna_study\", \"ResNet_Enhanced\")\n        \n        # === INFORMACIÓN DEL SISTEMA ===\n        mlflow.log_param(\"cpu_cores\", os.cpu_count())\n        mlflow.log_param(\"pytorch_version\", torch.__version__)\n        mlflow.log_param(\"cuda_available\", torch.cuda.is_available())\n        mlflow.log_param(\"mps_available\", torch.backends.mps.is_available())\n        \n        # === BUCLE DE ENTRENAMIENTO ===\n        for epoch in range(num_epochs):\n            epoch_start_time = time.time()\n            print(f'\\n📊 ÉPOCA {epoch+1}/{num_epochs}')\n            print('-' * 50)\n            \n            # Métricas por época\n            epoch_metrics = {\n                'train_loss': 0.0,\n                'train_acc': 0.0,\n                'val_loss': 0.0,\n                'val_acc': 0.0,\n                'val_kappa': 0.0\n            }\n            \n            # Variables para kappa\n            epoch_kappa_true = []\n            epoch_kappa_pred = []\n            epoch_output_scores = []\n            \n            # === FASES: TRAIN Y VALIDATION ===\n            for phase in ['train', 'val']:\n                phase_start_time = time.time()\n                \n                if phase == 'train':\n                    model.train()\n                    print(\"🏋️ Fase de entrenamiento...\")\n                else:\n                    model.eval()\n                    print(\"🧪 Fase de validación...\")\n                \n                running_loss = 0.0\n                running_corrects = 0\n                \n                # Procesar batches\n                for inputs, labels in tqdm(dataloaders[phase], desc=f\"{phase.title()}\"):\n                    inputs = inputs.to(device)\n                    labels = labels.to(device)\n                    \n                    optimizer.zero_grad()\n                    \n                    with torch.set_grad_enabled(phase == 'train'):\n                        outputs = model(inputs)\n                        _, preds = torch.max(outputs, 1)\n                        loss = criterion(outputs, labels)\n                        \n                        if phase == 'train':\n                            loss.backward()\n                            optimizer.step()\n                        else:\n                            # Guardar para kappa\n                            epoch_kappa_true.extend(labels.cpu().numpy().tolist())\n                            epoch_kappa_pred.extend(preds.cpu().numpy().tolist())\n                            outputs_np = outputs.cpu().numpy()\n                            epoch_output_scores.extend([outputs_np[i,:] for i in range(outputs_np.shape[0])])\n                    \n                    running_loss += loss.item() * inputs.size(0)\n                    running_corrects += torch.sum(preds == labels.data)\n                \n                # Calcular métricas de la fase\n                epoch_loss = running_loss / len(datasets[phase])\n                if device.type == 'mps':\n                    epoch_acc = running_corrects.float() / len(datasets[phase])\n                else:\n                    epoch_acc = running_corrects.double() / len(datasets[phase])\n                \n                # Calcular kappa para validación\n                current_kappa = np.nan\n                if phase == 'val':\n                    current_kappa = cohen_kappa_score(epoch_kappa_true, epoch_kappa_pred, weights='quadratic')\n                \n                # Guardar métricas\n                epoch_metrics[f'{phase}_loss'] = epoch_loss\n                epoch_metrics[f'{phase}_acc'] = float(epoch_acc)\n                if phase == 'val':\n                    epoch_metrics['val_kappa'] = current_kappa\n                \n                phase_duration = time.time() - phase_start_time\n                \n                # === MLFLOW: Log métricas por fase ===\n                mlflow.log_metric(f\"{phase}_loss\", epoch_loss, step=epoch)\n                mlflow.log_metric(f\"{phase}_accuracy\", float(epoch_acc), step=epoch)\n                mlflow.log_metric(f\"{phase}_duration_seconds\", phase_duration, step=epoch)\n                \n                if not np.isnan(current_kappa):\n                    mlflow.log_metric(f\"{phase}_kappa\", current_kappa, step=epoch)\n                \n                print(f'{phase.title()} - Loss: {epoch_loss:.4f}, Acc: {epoch_acc*100:.2f}%, Kappa: {current_kappa:.4f}')\n                \n                # === GUARDAR MEJOR MODELO ===\n                if phase == 'val' and current_kappa > best_kappa:\n                    best_acc = epoch_acc\n                    best_kappa = current_kappa\n                    best_model_wts = copy.deepcopy(model.state_dict())\n                    \n                    print(f\"🏆 ¡NUEVO MEJOR MODELO! Kappa: {best_kappa:.6f}\")\n                    \n                    # === MLFLOW: Log mejores métricas ===\n                    mlflow.log_metric(\"best_accuracy\", float(best_acc))\n                    mlflow.log_metric(\"best_kappa\", best_kappa)\n                    mlflow.log_metric(\"best_epoch\", epoch)\n                    \n                    # === GUARDAR ARTEFACTOS SI ES MEJOR QUE TRIALS ANTERIORES ===\n                    if trial and best_kappa > previous_best:\n                        print(f\"🎯 Mejor que trials anteriores! (prev: {previous_best:.6f})\")\n                        \n                        # Guardar predicciones\n                        pred_filename = f\"predictions_trial_{trial.number}.joblib\"\n                        pred_path = os.path.join(\"/tmp\", pred_filename)\n                        predicted_df = pd.DataFrame({\n                            'PetID': test_sample_ids,\n                            'pred': epoch_output_scores\n                        })\n                        dump(predicted_df, pred_path)\n                        mlflow.log_artifact(pred_path, \"predictions\")\n                        \n                        # Guardar matriz de confusión\n                        cm_filename = f\"confusion_matrix_trial_{trial.number}.jpg\"\n                        cm_path = os.path.join(\"/tmp\", cm_filename)\n                        try:\n                            plot_confusion_matrix(epoch_kappa_true, epoch_kappa_pred).write_image(cm_path)\n                            mlflow.log_artifact(cm_path, \"confusion_matrices\")\n                        except:\n                            print(\"⚠️ No se pudo generar matriz de confusión\")\n            \n            # === MLFLOW: Log métricas del sistema por época ===\n            system_summary = system_monitor.get_summary_metrics()\n            for metric_name, value in system_summary.items():\n                mlflow.log_metric(f\"system_{metric_name}\", value, step=epoch)\n            \n            epoch_duration = time.time() - epoch_start_time\n            mlflow.log_metric(\"epoch_duration_seconds\", epoch_duration, step=epoch)\n            \n            print(f\"⏱️ Época completada en {epoch_duration:.1f}s\")\n        \n        # === FINALIZACIÓN DEL ENTRENAMIENTO ===\n        total_time = time.time() - since\n        system_monitor.stop_monitoring()\n        \n        # === MLFLOW: Métricas finales y sistema ===\n        mlflow.log_metric(\"total_training_time_seconds\", total_time)\n        mlflow.log_metric(\"total_training_time_minutes\", total_time/60)\n        mlflow.log_metric(\"final_best_accuracy\", float(best_acc))\n        mlflow.log_metric(\"final_best_kappa\", best_kappa)\n        \n        # Métricas finales del sistema\n        final_system_summary = system_monitor.get_summary_metrics()\n        for metric_name, value in final_system_summary.items():\n            mlflow.log_metric(f\"final_system_{metric_name}\", value)\n        \n        # === MLFLOW: Guardar modelo final ===\n        model.load_state_dict(best_model_wts)\n        \n        if trial and best_kappa > previous_best:\n            # Ejemplo de entrada para signatura\n            try:\n                dummy_input = torch.randn(1, 3, 299, 299).to(device)\n                mlflow.pytorch.log_model(\n                    model,\n                    \"best_model\",\n                    input_example=dummy_input.cpu().numpy()\n                )\n                print(\"💾 Modelo registrado en MLflow con signatura\")\n            except Exception as e:\n                print(f\"⚠️ Error registrando modelo: {e}\")\n        \n        # === TAGS FINALES ===\n        mlflow.set_tag(\"training_completed\", \"true\")\n        mlflow.set_tag(\"device_used\", str(device))\n        mlflow.set_tag(\"total_epochs\", num_epochs)\n        mlflow.set_tag(\"enhanced_version\", \"5.0\")\n        \n        print(f\"\\n🏁 ENTRENAMIENTO COMPLETADO\")\n        print(f\"⏱️ Tiempo total: {total_time//60:.0f}m {total_time%60:.0f}s\")\n        print(f\"🎯 Mejor accuracy: {best_acc*100:.2f}%\")\n        print(f\"🏆 Mejor kappa: {best_kappa:.6f}\")\n    \n    return model, best_kappa

In [ ]:
# ===== FUNCIÓN OPTUNA ENHANCED V5.0 =====
def optuna_train_enhanced(trial):\n    \"\"\"\n    Función de optimización Optuna con registro completo en MLflow v5.0\n    - Tracking detallado de hiperparámetros\n    - Métricas del sistema por trial\n    - Organización jerárquica de experimentos\n    \"\"\"\n    \n    print(f\"\\n🔬 OPTUNA TRIAL #{trial.number} - Enhanced v5.0\")\n    print(\"=\"*60)\n    \n    # Sugerir hiperparámetros\n    epochs = trial.suggest_int('epochs', ciclos, ciclos)\n    lr = trial.suggest_float('lr', 0.00001, 0.1, log=True)\n    momentum = trial.suggest_float('momentum', 0.0, 0.95)\n    \n    print(f\"📊 Hiperparámetros del trial:\")\n    print(f\"   ├── Epochs: {epochs}\")\n    print(f\"   ├── Learning Rate: {lr:.6f}\")\n    print(f\"   └── Momentum: {momentum:.4f}\")\n    \n    # ===== MLFLOW: Run padre para el trial =====\n    with mlflow.start_run(run_name=f\"optuna_trial_{trial.number}_enhanced\", nested=True) as parent_run:\n        \n        # === PARÁMETROS DEL TRIAL ===\n        mlflow.log_param(\"optuna_trial_number\", trial.number)\n        mlflow.log_param(\"suggested_epochs\", epochs)\n        mlflow.log_param(\"suggested_lr\", lr)\n        mlflow.log_param(\"suggested_momentum\", momentum)\n        mlflow.log_param(\"optuna_study_name\", \"ResNet_Enhanced_v5\")\n        \n        # === TAGS DEL TRIAL ===\n        mlflow.set_tag(\"optuna_trial\", \"true\")\n        mlflow.set_tag(\"trial_number\", trial.number)\n        mlflow.set_tag(\"enhanced_tracking\", \"true\")\n        mlflow.set_tag(\"version\", \"5.0\")\n        \n        # === INFORMACIÓN DEL ENTORNO ===\n        trial_start_time = time.time()\n        mlflow.log_param(\"trial_start_timestamp\", datetime.datetime.now().isoformat())\n        \n        # Ejecutar entrenamiento con parámetros sugeridos\n        _, best_score = train_val_enhanced(\n            resnet50, criterion,\n            dataloaders={'train': train_dataloader, 'val': val_dataloader},\n            datasets={'train': train_data, 'val': val_data},\n            device=device,\n            num_epochs=epochs,\n            lr=lr,\n            momentum=momentum,\n            trial=trial\n        )\n        \n        # === MÉTRICAS FINALES DEL TRIAL ===\n        trial_duration = time.time() - trial_start_time\n        mlflow.log_metric(\"trial_duration_seconds\", trial_duration)\n        mlflow.log_metric(\"trial_duration_minutes\", trial_duration/60)\n        mlflow.log_metric(\"trial_final_kappa\", best_score)\n        \n        # === TAGS FINALES ===\n        mlflow.set_tag(\"trial_completed\", \"true\")\n        mlflow.set_tag(\"trial_end_timestamp\", datetime.datetime.now().isoformat())\n        \n        print(f\"\\n🏁 TRIAL #{trial.number} COMPLETADO\")\n        print(f\"⏱️ Duración: {trial_duration//60:.0f}m {trial_duration%60:.0f}s\")\n        print(f\"🎯 Kappa final: {best_score:.6f}\")\n        print(\"=\"*60)\n    \n    return best_score\n\n# ===== CONFIGURACIÓN Y EJECUCIÓN DE OPTUNA ENHANCED =====\nprint(\"\\n🔍 CONFIGURANDO OPTIMIZACIÓN OPTUNA ENHANCED v5.0\")\nprint(f\"📊 Proyecto: {proyecto}\")\nprint(f\"🔬 Experimento: {experimento}\")\nprint(f\"🎯 Objetivo: Maximizar Kappa\")\nprint(f\"🔄 Número de trials: 20\")\n\n# Configurar MLflow para Optuna\nmlflow_tracking_uri = f\"file://{os.path.abspath('../work')}/mlruns\"\nmlflow.set_tracking_uri(mlflow_tracking_uri)\n\nexperiment_name = f\"{proyecto}_{experimento}\"\ntry:\n    experiment = mlflow.create_experiment(experiment_name)\n    print(f\"🆕 Experimento MLflow creado: {experiment_name}\")\nexcept mlflow.exceptions.MlflowException:\n    experiment = mlflow.get_experiment_by_name(experiment_name)\n    print(f\"📂 Experimento MLflow existente: {experiment_name}\")\n\nmlflow.set_experiment(experiment_name)\nprint(f\"✅ MLflow configurado: {mlflow_tracking_uri}\")\n\nprint(\"\\n🚀 ¡CONFIGURACIÓN COMPLETA! Listo para entrenar con tracking enhanced v5.0\")"

In [ ]:
# ===== CALLBACK DE PARADA TEMPRANA V5.0 =====
class EarlyStoppingCallback:
    """
    Callback para detener Optuna después de N trials consecutivos sin mejoras
    """
    
    def __init__(self, patience=5, min_improvement=0.001):
        self.patience = patience  # Número de trials sin mejora antes de parar
        self.min_improvement = min_improvement  # Mejora mínima requerida
        self.trials_without_improvement = 0
        self.best_value = None
        self.best_trial = None
        
    def __call__(self, study, trial):
        current_value = trial.value
        
        # Si es el primer trial o hay mejora significativa
        if self.best_value is None or current_value > (self.best_value + self.min_improvement):
            self.best_value = current_value
            self.best_trial = trial.number
            self.trials_without_improvement = 0
            
            print(f"🎯 MEJORA DETECTADA! Trial {trial.number}")
            print(f"   ├── Nuevo mejor valor: {current_value:.6f}")
            print(f"   ├── Mejora de: {current_value - (self.best_value - current_value + self.min_improvement):.6f}")
            print(f"   └── Trials sin mejora: 0")
            
            # Log en MLflow
            with mlflow.start_run(run_name=f"early_stopping_update_trial_{trial.number}", nested=True):
                mlflow.log_metric("best_value_updated", current_value)
                mlflow.log_metric("improvement", current_value - (self.best_value - current_value + self.min_improvement))
                mlflow.log_metric("trials_without_improvement", 0)
                mlflow.set_tag("improvement_detected", "true")
                mlflow.set_tag("best_trial", trial.number)
        else:
            self.trials_without_improvement += 1
            
            print(f"⚠️ Sin mejora en trial {trial.number}")
            print(f"   ├── Valor actual: {current_value:.6f}")
            print(f"   ├── Mejor valor: {self.best_value:.6f}")
            print(f"   ├── Trials sin mejora: {self.trials_without_improvement}/{self.patience}")
            print(f"   └── Diferencia: {self.best_value - current_value:.6f}")
            
            # Log en MLflow
            with mlflow.start_run(run_name=f"early_stopping_check_trial_{trial.number}", nested=True):
                mlflow.log_metric("current_value", current_value)
                mlflow.log_metric("best_value", self.best_value)
                mlflow.log_metric("trials_without_improvement", self.trials_without_improvement)
                mlflow.log_metric("difference_from_best", self.best_value - current_value)
                mlflow.set_tag("improvement_detected", "false")
                
                if self.trials_without_improvement >= self.patience:
                    mlflow.set_tag("early_stopping_triggered", "true")
        
        # Detener si se alcanza la paciencia
        if self.trials_without_improvement >= self.patience:
            print(f"\\n🛑 PARADA TEMPRANA ACTIVADA!")
            print(f"   ├── Trials consecutivos sin mejora: {self.trials_without_improvement}")
            print(f"   ├── Paciencia configurada: {self.patience}")
            print(f"   ├── Mejor trial: #{self.best_trial}")
            print(f"   ├── Mejor valor: {self.best_value:.6f}")
            print(f"   └── Deteniendo optimización...")
            
            # Log final en MLflow
            with mlflow.start_run(run_name="early_stopping_triggered", nested=True):
                mlflow.log_metric("final_trials_without_improvement", self.trials_without_improvement)
                mlflow.log_metric("patience_limit", self.patience)
                mlflow.log_metric("final_best_value", self.best_value)
                mlflow.log_param("final_best_trial", self.best_trial)
                mlflow.set_tag("early_stopping_reason", "patience_exceeded")
                mlflow.set_tag("optimization_stopped", "true")
            
            study.stop()

# Crear callback de parada temprana
early_stopping = EarlyStoppingCallback(patience=5, min_improvement=0.001)

print("✅ Sistema de parada temprana configurado")
print(f"   ├── Paciencia: 5 trials consecutivos sin mejora")
print(f"   ├── Mejora mínima requerida: 0.001")
print(f"   └── Se detendrá automáticamente si no hay progreso")

In [ ]:
# ===== EJECUCIÓN DE OPTUNA CON PARADA TEMPRANA V5.0 =====

# Configurar todos los parámetros necesarios
MODEL_NAME = '05 ResNet Enhanced'
MODEL_VERSION = '5.0.0'

# Crear directorio de trabajo si no existe
work_dir = f"../work/{proyecto}"
os.makedirs(work_dir, exist_ok=True)

print("🔍 INICIANDO OPTIMIZACIÓN OPTUNA CON PARADA TEMPRANA")
print(f"   ├── Proyecto: {proyecto}")
print(f"   ├── Experimento: {experimento}")
print(f"   ├── Modelo: {MODEL_NAME} v{MODEL_VERSION}")
print(f"   ├── Base de datos: {work_dir}/db.sqlite3")
print(f"   ├── Máximo de trials: 20")
print(f"   └── Parada temprana: 5 trials sin mejora")

# Crear estudio Optuna
study = optuna.create_study(
    direction='maximize',
    storage=f"sqlite:///{work_dir}/db.sqlite3",
    study_name=f'{MODEL_NAME}_{MODEL_VERSION}_early_stopping',
    load_if_exists=True
)

# Configurar paths para artefactos
PATH_TO_TEMP_FILES = os.path.join(work_dir, "optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(work_dir, "optuna_artifacts")
os.makedirs(PATH_TO_TEMP_FILES, exist_ok=True)
os.makedirs(PATH_TO_OPTUNA_ARTIFACTS, exist_ok=True)

# Configurar FileSystemArtifactStore para Optuna
artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)

# ===== MLFLOW: Run principal del estudio =====
with mlflow.start_run(run_name=f"optuna_study_early_stopping_{MODEL_NAME}_{MODEL_VERSION}") as study_run:
    
    # === PARÁMETROS DEL ESTUDIO ===
    mlflow.log_param("max_trials", 20)
    mlflow.log_param("direction", "maximize")
    mlflow.log_param("study_name", f'{MODEL_NAME}_{MODEL_VERSION}_early_stopping')
    mlflow.log_param("storage", f"sqlite:///{work_dir}/db.sqlite3")
    mlflow.log_param("early_stopping_patience", 5)
    mlflow.log_param("min_improvement", 0.001)
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("model_version", MODEL_VERSION)
    
    # === TAGS DEL ESTUDIO ===
    mlflow.set_tag("optuna_study", "true")
    mlflow.set_tag("early_stopping", "enabled")
    mlflow.set_tag("enhanced_version", "5.0")
    mlflow.set_tag("study_type", "hyperparameter_optimization")
    
    # === INFORMACIÓN DEL SISTEMA ===
    mlflow.log_param("device_type", str(device))
    mlflow.log_param("pytorch_version", torch.__version__)
    mlflow.log_param("cpu_cores", os.cpu_count())
    
    study_start_time = time.time()
    mlflow.log_param("study_start_timestamp", datetime.datetime.now().isoformat())
    
    print("\\n🚀 Comenzando optimización con parada temprana...")
    print("=" * 70)
    
    try:
        # EJECUTAR OPTIMIZACIÓN
        study.optimize(
            optuna_train_enhanced, 
            n_trials=20, 
            callbacks=[early_stopping]
        )
        
        optimization_status = "completed_normally"
        if study.trials and early_stopping.trials_without_improvement >= early_stopping.patience:
            optimization_status = "stopped_early"
            
    except optuna.exceptions.StudyStopException:
        optimization_status = "stopped_early"
        print("\\n✅ Optimización detenida por parada temprana")
        
    except Exception as e:
        optimization_status = "error"
        print(f"\\n❌ Error durante optimización: {e}")
        mlflow.log_param("error_message", str(e))
    
    # === RESULTADOS FINALES ===
    study_duration = time.time() - study_start_time
    
    print("\\n" + "=" * 70)
    print("🎯 OPTIMIZACIÓN COMPLETADA")
    print(f"   ├── Estado: {optimization_status}")
    print(f"   ├── Duración total: {study_duration//60:.0f}m {study_duration%60:.0f}s")
    print(f"   ├── Trials completados: {len(study.trials)}")
    
    if study.trials:
        print(f"   ├── Mejor trial: #{study.best_trial.number}")
        print(f"   ├── Mejor kappa: {study.best_value:.6f}")
        print(f"   └── Mejores parámetros: {study.best_params}")
        
        # === MLFLOW: Log resultados finales ===
        mlflow.log_metric("study_duration_seconds", study_duration)
        mlflow.log_metric("study_duration_minutes", study_duration/60)
        mlflow.log_metric("total_trials_completed", len(study.trials))
        mlflow.log_metric("best_trial_number", study.best_trial.number)
        mlflow.log_metric("best_study_kappa", study.best_value)
        
        # Log mejores parámetros
        for param_name, param_value in study.best_params.items():
            mlflow.log_param(f"best_{param_name}", param_value)
            
        # Log información de parada temprana
        mlflow.log_metric("final_trials_without_improvement", early_stopping.trials_without_improvement)
        mlflow.log_metric("early_stopping_patience", early_stopping.patience)
        
    else:
        print("   └── No se completaron trials")
        mlflow.log_metric("total_trials_completed", 0)
    
    # === TAGS FINALES ===
    mlflow.set_tag("study_completed", "true")
    mlflow.set_tag("optimization_status", optimization_status)
    mlflow.set_tag("study_end_timestamp", datetime.datetime.now().isoformat())
    
    if optimization_status == "stopped_early":
        mlflow.set_tag("early_stopping_triggered", "true")
        mlflow.set_tag("reason", "no_improvement_for_patience_trials")
    
    print("=" * 70)

print("\\n✅ Estudio completado con parada temprana configurada")
print("📊 Revisa MLflow UI para análisis detallado de trials y métricas")

In [4]:
# ===== ANÁLISIS DE RESULTADOS CON PARADA TEMPRANA =====

def analyze_early_stopping_results(study, early_stopping_callback):
    """
    Analiza los resultados del estudio con parada temprana
    """
    
    print("📊 ANÁLISIS DE RESULTADOS CON PARADA TEMPRANA")
    print("=" * 60)
    
    if not study.trials:
        print("❌ No hay trials para analizar")
        return
    
    # Información básica
    total_trials = len(study.trials)
    completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    
    print(f"📈 ESTADÍSTICAS GENERALES:")
    print(f"   ├── Total trials: {total_trials}")
    print(f"   ├── Trials completados: {len(completed_trials)}")
    print(f"   ├── Trials fallidos: {total_trials - len(completed_trials)}")
    print(f"   └── Tasa de éxito: {len(completed_trials)/total_trials*100:.1f}%")
    
    # Información de parada temprana
    print(f"\\n🛑 INFORMACIÓN DE PARADA TEMPRANA:")
    print(f"   ├── Paciencia configurada: {early_stopping_callback.patience}")
    print(f"   ├── Mejora mínima requerida: {early_stopping_callback.min_improvement}")
    print(f"   ├── Mejor trial encontrado: #{early_stopping_callback.best_trial}")
    print(f"   ├── Mejor valor: {early_stopping_callback.best_value:.6f}")
    print(f"   └── Trials sin mejora al final: {early_stopping_callback.trials_without_improvement}")
    
    # ¿Se activó la parada temprana?
    early_stopped = early_stopping_callback.trials_without_improvement >= early_stopping_callback.patience
    print(f"\\n⚡ ESTADO DE PARADA TEMPRANA:")
    if early_stopped:
        print(f"   ✅ ACTIVADA - Se detuvo tras {early_stopping_callback.patience} trials sin mejora")
        print(f"   💡 Ahorro estimado: {20 - total_trials} trials no ejecutados")
        print(f"   ⏱️  Tiempo ahorrado: ~{(20 - total_trials) * 10:.0f} minutos estimados")
    else:
        print(f"   ❌ NO ACTIVADA - Se completaron todos los trials planificados")
    
    # Análisis de convergencia
    if len(completed_trials) > 1:
        values = [t.value for t in completed_trials]
        
        print(f"\\n📊 ANÁLISIS DE CONVERGENCIA:")
        print(f"   ├── Mejor kappa: {max(values):.6f}")
        print(f"   ├── Peor kappa: {min(values):.6f}")
        print(f"   ├── Kappa promedio: {np.mean(values):.6f}")
        print(f"   ├── Desviación estándar: {np.std(values):.6f}")
        print(f"   └── Rango: {max(values) - min(values):.6f}")
        
        # Progreso por trials
        print(f"\\n🎯 EVOLUCIÓN DEL MEJOR VALOR:")
        best_so_far = []
        current_best = -float('inf')
        
        for i, trial in enumerate(completed_trials):
            if trial.value > current_best:
                current_best = trial.value
                print(f"   ├── Trial {trial.number}: {trial.value:.6f} ⬆️ (MEJORA)")
            else:
                print(f"   ├── Trial {trial.number}: {trial.value:.6f}")
            best_so_far.append(current_best)
    
    # Recomendaciones
    print(f"\\n💡 RECOMENDACIONES:")
    if early_stopped:
        print(f"   ✅ Parada temprana funcionó correctamente")
        print(f"   📊 El modelo convergió en {early_stopping_callback.best_trial} trials")
        print(f"   🎯 Valor óptimo: {early_stopping_callback.best_value:.6f}")
    else:
        print(f"   🔄 Considera aumentar el número máximo de trials")
        print(f"   ⚙️  O ajustar la paciencia/mejora mínima")
    
    print("=" * 60)

# Ejecutar análisis si hay un estudio disponible
try:
    analyze_early_stopping_results(study, early_stopping)
    
    # Log del análisis en MLflow
    with mlflow.start_run(run_name="early_stopping_analysis", nested=True):
        mlflow.log_metric("analysis_total_trials", len(study.trials))
        mlflow.log_metric("analysis_completed_trials", len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]))
        mlflow.log_metric("analysis_early_stopped", int(early_stopping.trials_without_improvement >= early_stopping.patience))
        mlflow.log_metric("analysis_best_value", early_stopping.best_value if early_stopping.best_value else 0)
        mlflow.log_param("analysis_best_trial", early_stopping.best_trial if early_stopping.best_trial else -1)
        mlflow.set_tag("analysis_type", "early_stopping_summary")
        
except NameError:
    print("⚠️ Ejecuta primero el estudio de Optuna para ver el análisis")
except Exception as e:
    print(f"❌ Error en análisis: {e}")

print("\\n🎉 NOTEBOOK v5.0 CON PARADA TEMPRANA COMPLETADO")
print("🔬 Características implementadas:")
print("   ├── ✅ Parada temprana tras 5 trials sin mejora")
print("   ├── ✅ Registro completo en MLflow")
print("   ├── ✅ Monitoreo de recursos del sistema")
print("   ├── ✅ Métricas detalladas por época y trial")
print("   ├── ✅ Artefactos organizados")
print("   └── ✅ Análisis automático de resultados")

⚠️ Ejecuta primero el estudio de Optuna para ver el análisis
\n🎉 NOTEBOOK v5.0 CON PARADA TEMPRANA COMPLETADO
🔬 Características implementadas:
   ├── ✅ Parada temprana tras 5 trials sin mejora
   ├── ✅ Registro completo en MLflow
   ├── ✅ Monitoreo de recursos del sistema
   ├── ✅ Métricas detalladas por época y trial
   ├── ✅ Artefactos organizados
   └── ✅ Análisis automático de resultados
